# NEAT Training Notebook: CO2 Strategy Optimization

This notebook trains a NEAT (NeuroEvolution of Augmenting Topologies) policy to minimize CO2 emissions by selecting the best reuse strategy under simulated plant conditions.

## Notebook flow
1. Load and preprocess simulation states
2. Define a physics-based CO2 simulator
3. Define genome fitness evaluation
4. Run neuroevolution and save the winner


In [1]:
import neat
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import pickle
import random
from sklearn.preprocessing import MinMaxScaler

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

STRATEGIES = (
    "Biomass combustion",
    "Animal feed",
    "Composting",
    "Biochar",
)

RAW_REQUIRED_COLUMNS = (
    "generated_volume_tons",
    "moisture_pct",
    "process_temperature_c",
    "subproduct_type",
    "season",
)

INPUT_COLUMNS = (
    "generated_volume_tons",
    "moisture_pct",
    "process_temperature_c",
    "subproduct_type_Husk",
    "subproduct_type_Straw",
    "subproduct_type_Silo dust",
    "subproduct_type_Bran",
    "season_Rainy",
    "season_Dry",
)

PREPROCESSED_CATEGORY_COLUMNS = (
    "subproduct_type_Husk",
    "subproduct_type_Straw",
    "subproduct_type_Silo dust",
    "subproduct_type_Bran",
    "season_Rainy",
    "season_Dry",
)

## 1. Load and Preprocess Simulation States

We load the synthetic training dataset, sample 1,000 representative plant states, and prepare the 9 input features expected by the NEAT network:
- 3 scaled physical inputs: `generated_volume_tons`, `moisture_pct`, `process_temperature_c`
- 4 one-hot subproduct features
- 2 one-hot season features

In [2]:
def validate_schema(df: pd.DataFrame) -> None:
    has_raw_schema = all(col in df.columns for col in RAW_REQUIRED_COLUMNS)
    has_preprocessed_schema = all(col in df.columns for col in INPUT_COLUMNS)

    if not has_raw_schema and not has_preprocessed_schema:
        raise ValueError("Dataset must contain either raw columns or preprocessed feature columns.")

    candidate_columns = RAW_REQUIRED_COLUMNS if has_raw_schema else INPUT_COLUMNS
    null_counts = df[list(candidate_columns)].isna().sum()
    if int(null_counts.sum()) > 0:
        raise ValueError(f"Null values found in required columns: {null_counts.to_dict()}")


def is_preprocessed(df: pd.DataFrame) -> bool:
    return all(col in df.columns for col in INPUT_COLUMNS) and not all(
        col in df.columns for col in RAW_REQUIRED_COLUMNS
    )


def resolve_scaler_path_for_scaled_dataset(dataset_path: Path) -> Path:
    dataset_name = dataset_path.name
    suffixes = ("_train_scaled.csv", "_test_scaled.csv")
    prefix = None
    for suffix in suffixes:
        if dataset_name.endswith(suffix):
            prefix = dataset_name[: -len(suffix)]
            break

    if prefix is None:
        raise ValueError(
            "Cannot infer scaler path from scaled dataset name. "
            "Expected suffix '_train_scaled.csv' or '_test_scaled.csv'."
        )

    scaler_path = dataset_path.parent / f"{prefix}_scaler.joblib"
    if not scaler_path.exists():
        raise FileNotFoundError(f"Scaler artifact not found: {scaler_path}")
    return scaler_path


def recover_residue_labels(df: pd.DataFrame, residue_columns: list[str]) -> pd.Series:
    residue_frame = df[residue_columns]
    return residue_frame.idxmax(axis=1).str.replace("subproduct_type_", "", regex=False)


def prepare_features(
    df: pd.DataFrame,
    sample_size: int,
    random_state: int,
    dataset_path: Path,
 ):
    sampled_df = df.sample(n=sample_size, random_state=random_state).copy()

    if is_preprocessed(sampled_df):
        features_df = sampled_df[list(INPUT_COLUMNS)].astype(float)
        residue_labels = recover_residue_labels(sampled_df, list(PREPROCESSED_CATEGORY_COLUMNS[:4]))
        scaler_path = resolve_scaler_path_for_scaled_dataset(dataset_path)
        scaler = joblib.load(scaler_path)
        physical_columns = [
            "generated_volume_tons",
            "moisture_pct",
            "process_temperature_c",
        ]
        physical_ranges = {
            column: (float(scaler.data_min_[idx]), float(scaler.data_max_[idx]))
            for idx, column in enumerate(physical_columns)
        }
        return (
            features_df.reset_index(drop=True),
            sampled_df.reset_index(drop=True),
            residue_labels.reset_index(drop=True),
            physical_ranges,
        )

    residue_labels = sampled_df["subproduct_type"].astype(str)
    encoded_df = pd.get_dummies(sampled_df, columns=["subproduct_type", "season"])

    scaler = MinMaxScaler()
    physical_columns = [
        "generated_volume_tons",
        "moisture_pct",
        "process_temperature_c",
    ]
    encoded_df[physical_columns] = scaler.fit_transform(encoded_df[physical_columns])

    for expected_column in INPUT_COLUMNS:
        if expected_column not in encoded_df.columns:
            encoded_df[expected_column] = 0

    features_df = encoded_df[list(INPUT_COLUMNS)].astype(float)
    physical_ranges = {
        column: (float(scaler.data_min_[idx]), float(scaler.data_max_[idx]))
        for idx, column in enumerate(physical_columns)
    }
    return (
        features_df.reset_index(drop=True),
        sampled_df.reset_index(drop=True),
        residue_labels.reset_index(drop=True),
        physical_ranges,
    )


candidate_paths = [
    Path("../../data/split/dataset_optimization_cereal_co2_train_scaled.csv"),
    Path("../data/split/dataset_optimization_cereal_co2_train_scaled.csv"),
    Path("data/split/dataset_optimization_cereal_co2_train_scaled.csv"),
]

dataset_path = next((path for path in candidate_paths if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Could not locate dataset_optimization_cereal_co2_train_scaled.csv from current working directory."
    )

df = pd.read_csv(dataset_path)
validate_schema(df)

features_df, evaluation_df, residue_labels, physical_ranges = prepare_features(
    df=df,
    sample_size=500,
    random_state=SEED,
    dataset_path=dataset_path,
)

print(f"Dataset path: {dataset_path.resolve()}")
print(f"Original shape: {df.shape}")
print(f"Features shape: {features_df.shape}")
print(f"Evaluation shape: {evaluation_df.shape}")

Dataset path: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_train_scaled.csv
Original shape: (40000, 12)
Features shape: (500, 9)
Evaluation shape: (500, 12)


## 2. Physics-Based CO2 Simulator

The fitness signal uses a process-inspired approximation:
- Invert normalization to recover physical units
- Compute base thermal emissions
- Apply strategy-specific penalties/benefits according to moisture and material suitability

In [3]:
CAPACITY_TEMPLATE = {
    "Animal feed": 50.0,
    "Composting": 100.0,
    "Biochar": 15.0,
    "Biomass combustion": 5000.0,
}
LOTS_PER_DAY = 15

def denormalize_minmax(value_norm: float, min_value: float, max_value: float) -> float:
    normalized = float(np.clip(value_norm, 0.0, 1.0))
    return normalized * (max_value - min_value) + min_value


def calculate_simulated_co2(
    volume_norm: float,
    humidity_norm: float,
    temperature_norm: float,
    selected_strategy: str,
    residue_type: str,
 ) -> float:
    volume_min, volume_max = physical_ranges["generated_volume_tons"]
    humidity_min, humidity_max = physical_ranges["moisture_pct"]
    temperature_min, temperature_max = physical_ranges["process_temperature_c"]

    vol_real = denormalize_minmax(volume_norm, volume_min, volume_max)
    hum_real = denormalize_minmax(humidity_norm, humidity_min, humidity_max)
    temp_real = denormalize_minmax(temperature_norm, temperature_min, temperature_max)
    emission_base = (temp_real * 0.5) * vol_real

    if selected_strategy == "Biomass combustion":
        return emission_base + ((hum_real ** 1.5) * 2.0) - 50.0
    if selected_strategy == "Animal feed":
        if "Husk" in residue_type or "Straw" in residue_type or hum_real > 18.0:
            return emission_base * 1.8
        return emission_base * 0.4
    if selected_strategy == "Biochar":
        if hum_real < 10.0:
            return emission_base * 0.2
        return emission_base * 1.2
    if selected_strategy == "Composting":
        return emission_base * 0.8 + (vol_real * 5.0)
    return emission_base


def canonical_strategy(raw_strategy: str) -> str:
    normalized = raw_strategy.strip().lower()
    strategy_map = {
        "combustion biomasa": "Biomass combustion",
        "biomass combustion": "Biomass combustion",
        "alimentacion animal": "Animal feed",
        "animal feed": "Animal feed",
        "compostaje": "Composting",
        "composting": "Composting",
        "biochar": "Biochar",
    }
    if normalized not in strategy_map:
        raise ValueError(f"Unknown strategy label: {raw_strategy}")
    return strategy_map[normalized]


def reset_capacities() -> dict[str, float]:
    return dict(CAPACITY_TEMPLATE)


def select_feasible_strategy(
    scores: np.ndarray,
    volume_ton: float,
    current_capacities: dict[str, float],
 ) -> str:
    ranking = np.argsort(scores)[::-1]
    for strategy_index in ranking:
        strategy = STRATEGIES[int(strategy_index)]
        if volume_ton <= current_capacities.get(strategy, 0.0):
            current_capacities[strategy] -= volume_ton
            return strategy
    return "Biomass combustion"


def best_feasible_strategy(
    volume_norm: float,
    humidity_norm: float,
    temperature_norm: float,
    residue_type: str,
    volume_ton: float,
    current_capacities: dict[str, float],
 ) -> tuple[str, float]:
    best_strategy = "Biomass combustion"
    best_emissions = float("inf")

    for strategy in STRATEGIES:
        if volume_ton > current_capacities.get(strategy, 0.0):
            continue

        emissions = calculate_simulated_co2(
            volume_norm=volume_norm,
            humidity_norm=humidity_norm,
            temperature_norm=temperature_norm,
            selected_strategy=strategy,
            residue_type=residue_type,
        )
        if emissions < best_emissions:
            best_emissions = emissions
            best_strategy = strategy

    if best_emissions == float("inf"):
        best_emissions = calculate_simulated_co2(
            volume_norm=volume_norm,
            humidity_norm=humidity_norm,
            temperature_norm=temperature_norm,
            selected_strategy="Biomass combustion",
            residue_type=residue_type,
        )
    return best_strategy, best_emissions

## 3. Fitness Function Definition

Each genome is evaluated with the same reward shaping used in `src/training/evolution.py`:
- Capacity-constrained strategy choice per lot
- Comparison against baseline strategy emissions
- Comparison against best feasible strategy emissions
- Asymmetric rewards/penalties for suboptimal behavior

Fitness is the average weighted reward over sampled scenarios.

In [4]:
def evaluate_genomes(genomes, config):
    for genome_id, genome in genomes:
        try:
            net = neat.nn.FeedForwardNetwork.create(genome, config)
            weighted_reward_accumulated = 0.0
            current_capacities = reset_capacities()

            for row_idx, row in features_df.iterrows():
                evaluation_row = evaluation_df.iloc[row_idx]
                inputs = row.to_numpy(dtype=float)
                output = net.activate(inputs)

                if row_idx % LOTS_PER_DAY == 0:
                    current_capacities = reset_capacities()

                capacities_before_decision = dict(current_capacities)

                volume_min, volume_max = physical_ranges["generated_volume_tons"]
                volume_ton_real = denormalize_minmax(
                    value_norm=float(inputs[0]),
                    min_value=volume_min,
                    max_value=volume_max,
                )

                selected_strategy = select_feasible_strategy(
                    scores=np.asarray(output, dtype=float),
                    volume_ton=volume_ton_real,
                    current_capacities=current_capacities,
                )

                baseline_strategy = canonical_strategy(str(evaluation_row["reuse_strategy"]))
                residue = str(residue_labels.iloc[row_idx])

                baseline_emissions = calculate_simulated_co2(
                    volume_norm=float(inputs[0]),
                    humidity_norm=float(inputs[1]),
                    temperature_norm=float(inputs[2]),
                    selected_strategy=baseline_strategy,
                    residue_type=residue,
                )

                _, best_feasible_emissions = best_feasible_strategy(
                    volume_norm=float(inputs[0]),
                    humidity_norm=float(inputs[1]),
                    temperature_norm=float(inputs[2]),
                    residue_type=residue,
                    volume_ton=volume_ton_real,
                    current_capacities=capacities_before_decision,
                )

                co2_emitted = calculate_simulated_co2(
                    volume_norm=float(inputs[0]),
                    humidity_norm=float(inputs[1]),
                    temperature_norm=float(inputs[2]),
                    selected_strategy=selected_strategy,
                    residue_type=residue,
                )

                delta_vs_baseline = baseline_emissions - co2_emitted
                delta_vs_best = best_feasible_emissions - co2_emitted

                if delta_vs_baseline > 0:
                    weighted_reward_accumulated += delta_vs_baseline
                else:
                    weighted_reward_accumulated += delta_vs_baseline * 2.0

                if delta_vs_best >= 0:
                    weighted_reward_accumulated += delta_vs_best * 0.5
                else:
                    weighted_reward_accumulated += delta_vs_best * 3.0

            genome.fitness = weighted_reward_accumulated / len(features_df)
        except Exception as exc:
            print(f"Error while evaluating genome_id={genome_id}: {exc}")
            genome.fitness = -1e12

## 4. Run Neuroevolution and Save Artifacts

Configure NEAT, run 25 generations, and persist the winner genome to:

`../../models/artifacts/winner_genome.pkl`

This artifact is later consumed by inference scripts for constrained strategy assignment.

In [5]:
def run_neuroevolution(config_file: str, generations: int = 50):
    random.seed(SEED)
    np.random.seed(SEED)

    config = neat.Config(
        neat.DefaultGenome,
        neat.DefaultReproduction,
        neat.DefaultSpeciesSet,
        neat.DefaultStagnation,
        config_file,
    )

    population = neat.Population(config)
    stats = neat.StatisticsReporter()
    population.add_reporter(stats)

    print("Starting NEAT evolution...")
    winner_genome = population.run(evaluate_genomes, generations)

    output_path = Path("../../models/artifacts/winner_genome.pkl")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as file_obj:
        pickle.dump(winner_genome, file_obj)

    print("\nEvolution completed.")
    print(f"Winner genome:\n{winner_genome}")
    print(f"Winner genome saved at: {output_path.resolve()}")
    return winner_genome


if __name__ == "__main__":
    run_neuroevolution("../../config/config-feedforward.txt", generations=25)

Starting NEAT evolution...

Evolution completed.
Winner genome:
Key: 2701
Fitness: -0.5271941171090297
Nodes:
	0 DefaultNodeGene(key=0, bias=-0.9464496076671998, response=1.0, activation=sigmoid, aggregation=sum, time_constant=1.0)
	1 DefaultNodeGene(key=1, bias=-0.22806719301897949, response=1.0, activation=sigmoid, aggregation=sum, time_constant=1.0)
	2 DefaultNodeGene(key=2, bias=-0.8108356626778076, response=1.0, activation=sigmoid, aggregation=sum, time_constant=1.0)
	3 DefaultNodeGene(key=3, bias=0.3755960085167132, response=1.0, activation=sigmoid, aggregation=sum, time_constant=1.0)
	380 DefaultNodeGene(key=380, bias=-0.5718375847393519, response=1.0, activation=sigmoid, aggregation=sum, time_constant=1.0)
Connections:
	DefaultConnectionGene(key=(-9, 0), innovation=33, weight=0.6223215114354528, enabled=True)
	DefaultConnectionGene(key=(-9, 1), innovation=34, weight=-0.03626454286388625, enabled=False)
	DefaultConnectionGene(key=(-9, 2), innovation=35, weight=-0.408859751111754